In [ ]:
"""
FlowCAM → EcoTaxa converter (single sample version)
---------------------------------------------------
This script processes a single FlowCAM sample folder containing:
 - one .lst file
 - corresponding image files

It converst .tiff images to "uncompressed" ones - maybe its just our problem, but newer compuer don't want to read ouor FlowCAM VS-IV .tiff images. 

It reads the sample metadata from an Excel file, injects it into the EcoTaxa
object table, and outputs a zipped EcoTaxa-compatible archive (.zip)
containing the .tsv file and images.

Before running:
1. Install dependencies:
   pip install morphocut pandas openpyxl and other libs if you encouner an error

2. Edit the paths below to match your setup.
"""
# ok so here it is - my final code, made after many hours of talking with chatGPT : )
# it should work, but the only for one sample per processing
# the paths to the sample, to the file for final product , and to the metadata excel file should be defined
# remmember to prepare proper excel sheet with metadata in EcoTaxa style (object_; sample_; acq_; etc)
#author: Adam Makatun

In [1]:
from pathlib import Path
import shutil
import tifffile

import pandas as pd
import os

import zipfile
from io import BytesIO

from morphocut import Pipeline
from morphocut.file import Find
from morphocut.integration.flowcam import FlowCamReader
from morphocut.image import RGB2Gray
from morphocut.contrib.ecotaxa import EcotaxaWriter
from morphocut.str import Format
from morphocut.stream import Progress
from morphocut.image import ImageProperties
from morphocut.contrib.zooprocess import CalculateZooProcessFeatures

In [8]:
# === USER SETTINGS ==========================================================
# --- set source and destination folders of .TIFF (If conversiuon is needed!!!) ---
src_folder = Path() #your original FlowCAM file 
dst_folder = Path() # where you want your converted files saved

# Path to the folder with one FlowCAM sample (contains .lst and images)
input_folder = Path() # if you converted images, than it is "dst_folder" from above; if not, than it is your original FlowCAM file

# Path to the metadata Excel file
metadata_file = Path() #path to the excel file with the metadata, if you want to add it to .tsv file before uploading it to EcoTaxa

# Output .zip file path (will contain EcoTaxa .tsv + images)
output_file = Path() # where do you want to save .zip file? IMPORTANT!!!: must end with the name of the new zip file e.g. ".../data/myfile.zip" 

# ============================================================================




In [4]:


# --- copy whole folder tree ---
if dst_folder.exists():
    print(f"Destination already exists: {dst_folder}")
else:
    shutil.copytree(src_folder, dst_folder)
    print(f"Copied folder to: {dst_folder}")

# --- convert all TIFFs in copied folder to uncompressed TIFFs ---
tif_files = list(dst_folder.rglob("*.tif"))
print(f"Found {len(tif_files)} TIFF files")

for i, tif_path in enumerate(tif_files, 1):
    print(f"[{i}/{len(tif_files)}] Converting {tif_path.name} ...", flush=True)
    
    arr = tifffile.imread(tif_path)
    tmp_path = tif_path.with_suffix(".tmp.tif")
    
    if arr.ndim == 3 and arr.shape[-1] in (3, 4):
        tifffile.imwrite(tmp_path, arr, compression=None, photometric="rgb")
    else:
        tifffile.imwrite(tmp_path, arr, compression=None)
    
    tmp_path.replace(tif_path)

print("✅ Conversion finished")

Copied folder to: C:\Users\Adam\OneDrive - University of Gdansk\FlowCAM\Mytilus_tests\Try_1\Try_1_converted
Found 11 TIFF files
[1/11] Converting cal_image_000001.tif ...
[2/11] Converting Try_1_000001.tif ...
[3/11] Converting Try_1_000001_bin.tif ...
[4/11] Converting Try_1_000002.tif ...
[5/11] Converting Try_1_000002_bin.tif ...
[6/11] Converting Try_1_000003.tif ...
[7/11] Converting Try_1_000003_bin.tif ...
[8/11] Converting Try_1_000004.tif ...
[9/11] Converting Try_1_000004_bin.tif ...
[10/11] Converting Try_1_000005.tif ...
[11/11] Converting Try_1_000005_bin.tif ...
✅ Conversion finished


In [9]:


# your custom nodes
# from ... import ImageProperties, CalculateZooProcessFeatures

with Pipeline() as p:
    lst_fn = Find(str(input_folder), [".lst"])
    obj = FlowCamReader(lst_fn)

    img = obj.image
    mask = obj.mask
    img_gray = RGB2Gray(img, True)

    object_meta = obj.data

    object_id = Format("{lst_name}_{id}", lst_name=obj.lst_name, _kwargs=object_meta)
    object_meta["id"] = object_id

    regionprops = ImageProperties(mask, img_gray)
    object_meta = CalculateZooProcessFeatures(regionprops, object_meta)

    EcotaxaWriter(
        str(output_file),
        [(Format("{object_id}.jpg", object_id=object_id), img)],
        object_meta=object_meta,
    )

    Progress("Processing objects")

p.run()

print(f"\n✅ Done! EcoTaxa archive created at:\n{output_file}")

Processing objects: 100%|██████████████████████████████████████████████████████████████| 615/615 [00:02<00:00, 209it/s]

EcotaxaWriter: Wrote 615 entries to ecotaxa_export.tsv.
EcotaxaWriter: Wrote 615 objects to C:\Users\Adam\OneDrive - University of Gdansk\FlowCAM\Mytilus_tests\Try_1\Try_1_ecotaxa.zip.

✅ Done! EcoTaxa archive created at:
C:\Users\Adam\OneDrive - University of Gdansk\FlowCAM\Mytilus_tests\Try_1\Try_1_ecotaxa.zip


In [10]:
# --- Load metadata and create a dictionary ----------------------------------
metadata_df = pd.read_excel(metadata_file)

# Ensure consistent column naming
metadata_df.columns = [str(c).strip() for c in metadata_df.columns]

# --- Detect .lst file and extract sample_id ---------------------------------
lst_files = list(input_folder.glob("*.lst"))
if len(lst_files) == 0:
    raise FileNotFoundError(f"No .lst file found in {input_folder}")
if len(lst_files) > 1:
    print(f"⚠️ Multiple .lst files found, using the first one: {lst_files[0].name}")

lst_file = lst_files[0]
sample_id = lst_file.stem  # filename without extension

# --- Extract matching metadata row ------------------------------------------
meta_row = metadata_df.loc[metadata_df["sample_id"] == sample_id]
if meta_row.empty:
    raise ValueError(f"Sample ID '{sample_id}' not found in metadata Excel.")
metadata_dict = meta_row.iloc[0].to_dict()

print(f"Processing sample: {sample_id}")
print(f"Matched metadata: {metadata_dict}")


Processing sample: Try_1
Matched metadata: {'sample_id': 'Try_1', 'sample_ship': 'WOiG', 'sample_operator': 'Adam Makatun', 'sample_sampling_gear': 'culture', 'sample_season': 'none', 'sample_total_volume_m3': 1, 'sample_concentrated_sample_volume_ml': 11, 'sample_dilution_factor': 1, 'acq_instrument': 'FlowCAM VS-IV', 'acq_celltype': 'FC300', 'acq_volume_ml': 4.9394, 'acq_imaged_volume_ml': 2.0541, 'object_lat': 54.50889, 'object_lon': 18.54139, 'acq_id': 1}


In [11]:
# Now we need to add the metadata to the .tsv file (unfortunately, I could not do that in the morphocut pipeline - somhow it is not working)

# Paths
zip_path = output_file

# Load Excel metadata and set sample_id as index
meta_df = pd.read_excel(metadata_file).set_index("sample_id")

In [12]:
# Open zip and read TSV
with zipfile.ZipFile(zip_path, 'r') as z:
    tsv_name = [f for f in z.namelist() if f.endswith(".tsv")][0]
    tsv_bytes = z.read(tsv_name)
    
    # Keep other files (images) in memory
    other_files = {f: z.read(f) for f in z.namelist() if f != tsv_name}

# Read TSV as text to preserve headers
lines = tsv_bytes.decode("utf-8").splitlines()
header = lines[0].split("\t")
types = lines[1].split("\t")
data_lines = lines[2:]

# Remove 'object_label' if present
if "object_label" in header:
    idx = header.index("object_label")
    header.pop(idx)
    types.pop(idx)
    data_lines = ["\t".join([v for i, v in enumerate(line.split("\t")) if i != idx]) for line in data_lines]

# Create DataFrame
df = pd.DataFrame([line.split("\t") for line in data_lines], columns=header)

# Extract sample_id from object_id (assumes format: sampleID_something)
sample_id = df['object_id'].iloc[0].rsplit("_", 1)[0]
print("Detected sample ID:", sample_id)

# Get Excel metadata for this sample
if sample_id not in meta_df.index:
    raise ValueError(f"Sample ID '{sample_id}' not found in metadata Excel")
new_metadata = meta_df.loc[sample_id].to_dict()

# Add sample_id as a new column
df['sample_id'] = sample_id
header.append('sample_id')
types.append("[t]")  # treat as string

# Add Excel metadata as new columns
for col, val in new_metadata.items():
    df[col] = str(val)
    header.append(col)
    types.append("[t]" if isinstance(val, str) else "[f]")

# Ensure all data are strings
df = df.astype(str)

# Write updated TSV back into a BytesIO
tsv_buffer = BytesIO()
tsv_buffer.write(("\t".join(header) + "\n").encode("utf-8"))
tsv_buffer.write(("\t".join(types) + "\n").encode("utf-8"))
for _, row in df.iterrows():
    tsv_buffer.write(("\t".join(row) + "\n").encode("utf-8"))
tsv_buffer.seek(0)

# Recreate zip with updated TSV and original images
with zipfile.ZipFile(zip_path, 'w') as z:
    # Add updated TSV
    z.writestr(tsv_name, tsv_buffer.read())
    # Add other original files
    for f, content in other_files.items():
        z.writestr(f, content)

print(f"✅ Updated TSV inside zip: {zip_path}")


Detected sample ID: Try_1
✅ Updated TSV inside zip: C:\Users\Adam\OneDrive - University of Gdansk\FlowCAM\Mytilus_tests\Try_1\Try_1_ecotaxa.zip
